# ⚡ PricePilot AI — Milestone 2
## ML Model Training · KPI Extraction · LLM Integration

**Submitted by:** Yuvraj Nandu Patil  
**Email:** yuvrajcet26@gmail.com

---

This notebook covers all 4 tasks of Milestone 2 in order:
1. **Task 1** — Train 5 ML models and compare them
2. **Task 2** — Extract 6 Business KPIs with charts
3. **Task 3** — Connect to Groq LLM via API Key
4. **Task 4** — Send KPI data to LLM and get AI insights

**Dataset:** Integrated Pricing & Demand (7,300 rows × 31 columns)


## 🔧 STEP 0 — Install Libraries
Run this cell first to install all required packages.

In [ ]:
# Install all required libraries
!pip install xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn joblib groq -q
print('✅ All libraries installed successfully!')

## 📂 STEP 1 — Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['font.family']      = 'DejaVu Sans'

print('✅ Libraries imported!')

# ── Upload your CSV file OR mount Google Drive ─────────────
# Option A: Upload directly
# from google.colab import files
# uploaded = files.upload()

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/PricePilot/integrated_pricing_demand_dataset.csv')

# Load dataset (update path if needed)
df = pd.read_csv('integrated_pricing_demand_dataset.csv', parse_dates=['date'])

# Add time-based columns
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['quarter']    = df['date'].dt.quarter

print(f'Dataset shape      : {df.shape}')
print(f'Date range         : {df["date"].min()} to {df["date"].max()}')
print(f'Missing values     : {df.isnull().sum().sum()}')
print(f'Duplicate rows     : {df.duplicated().sum()}')
print(f'Unique products    : {df["product_name"].nunique()}')
print(f'Unique categories  : {df["category"].nunique()}')
print(f'Total revenue      : ${df["revenue"].sum():,.2f}')
print(f'Total units sold   : {df["units_sold"].sum():,}')
print('\nFirst 3 rows:')
df.head(3)

## 🔢 STEP 2 — Data Preprocessing
Convert text columns to numbers so ML models can use them.

In [ ]:
# Encode categorical text columns to numbers
le_cat  = LabelEncoder()
le_chan = LabelEncoder()
le_dow  = LabelEncoder()

df['category_enc']    = le_cat.fit_transform(df['category'])
df['channel_enc']     = le_chan.fit_transform(df['sales_channel'])
df['day_of_week_enc'] = le_dow.fit_transform(df['day_of_week'])

print('Category encoding:', dict(zip(le_cat.classes_, le_cat.transform(le_cat.classes_))))
print('Channel encoding :', dict(zip(le_chan.classes_, le_chan.transform(le_chan.classes_))))

# ── Define features for PRICE PREDICTION ──────────────────
price_features = [
    'cost_price', 'base_msrp', 'discount_pct', 'is_promotion',
    'competitor_1_price', 'competitor_2_price', 'competitor_3_price',
    'comp_avg_price', 'comp_min_price', 'product_rating', 'rating_count',
    'stock_level', 'is_holiday', 'macro_economic_index',
    'category_enc', 'channel_enc', 'month', 'is_weekend'
]

# ── Define features for DEMAND FORECASTING ────────────────
demand_features = [
    'current_price', 'discount_pct', 'is_promotion',
    'comp_avg_price', 'price_diff_vs_comp_avg', 'product_rating',
    'stock_level', 'stockout_flag', 'is_holiday', 'macro_economic_index',
    'category_enc', 'channel_enc', 'month', 'is_weekend', 'day_of_week_enc'
]

X_price  = df[price_features];  y_price  = df['current_price']
X_demand = df[demand_features]; y_demand = df['units_sold']

# Split 80% train / 20% test
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_price,  y_price,  test_size=0.2, random_state=42)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_demand, y_demand, test_size=0.2, random_state=42)

print(f'\nPrice task  — Train: {len(Xp_tr):,} | Test: {len(Xp_te):,}')
print(f'Demand task — Train: {len(Xd_tr):,} | Test: {len(Xd_te):,}')
print(f'Price features : {len(price_features)}')
print(f'Demand features: {len(demand_features)}')

---
# 📊 TASK 2 — Business KPI Extraction
Extract and visualise 6 important business KPIs from the dataset.

In [ ]:
# ════════════════════════════════════════════════════════
# KPI 1: Revenue & Profit Summary
# ════════════════════════════════════════════════════════

total_rev    = df['revenue'].sum()
total_profit = df['gross_profit'].sum()
avg_margin   = df['profit_margin_pct'].mean()
total_units  = df['units_sold'].sum()
avg_price    = df['current_price'].mean()
avg_comp     = df['comp_avg_price'].mean()
avg_rating   = df['product_rating'].mean()

print('='*55)
print('  KPI DASHBOARD — PricePilot AI (7,300 rows)')
print('='*55)
print(f'  KPI 1 — Total Revenue       : ${total_rev:>15,.2f}')
print(f'  KPI 2 — Total Gross Profit  : ${total_profit:>15,.2f}')
print(f'  KPI 3 — Avg Gross Margin    : {avg_margin:>14.2f}%')
print(f'  KPI 4 — Total Units Sold    : {total_units:>15,}')
print(f'  KPI 5 — Avg Selling Price   : ${avg_price:>14.2f}')
print(f'         Competitor Avg Price : ${avg_comp:>14.2f}')
print(f'         Price Advantage      : ${avg_comp-avg_price:>14.2f} cheaper than comp')
print(f'  KPI 6 — Avg Product Rating  : {avg_rating:>14.2f} / 5.0')

In [ ]:
# ── KPI Chart 1: Monthly Revenue & Profit Trend ───────────
monthly = df.groupby(['month','month_name']).agg(
    revenue=('revenue','sum'),
    profit=('gross_profit','sum'),
    units=('units_sold','sum')
).reset_index().sort_values('month')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('KPI 1 & 2 — Revenue & Profit Trends (2025)', fontsize=13, fontweight='bold')

# Monthly revenue
x = range(12)
axes[0].plot(x, monthly['revenue']/1e6, 'o-', color='#6366f1', lw=2.5, label='Revenue', ms=6)
axes[0].plot(x, monthly['profit']/1e6,  's--',color='#10b981', lw=2,   label='Profit',  ms=5)
axes[0].fill_between(x, monthly['revenue']/1e6, alpha=0.1, color='#6366f1')
axes[0].set_xticks(x); axes[0].set_xticklabels(monthly['month_name'], rotation=30)
axes[0].set_ylabel('Amount ($M)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Monthly Revenue vs Profit ($M)')

# Quarterly
qtr = df.groupby('quarter').agg(rev=('revenue','sum'), prof=('gross_profit','sum')).reset_index()
x2 = range(4)
axes[1].bar([i-0.2 for i in x2], qtr['rev']/1e6,  0.4, label='Revenue', color='#6366f1aa')
axes[1].bar([i+0.2 for i in x2], qtr['prof']/1e6, 0.4, label='Profit',  color='#10b981aa')
axes[1].set_xticks(x2); axes[1].set_xticklabels(['Q1','Q2','Q3','Q4'])
axes[1].set_ylabel('Amount ($M)'); axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')
axes[1].set_title('Quarterly Revenue vs Profit')

plt.tight_layout(); plt.savefig('kpi_revenue.png', dpi=150, bbox_inches='tight'); plt.show()
print('Peak quarter: Q4 at $', round(qtr["rev"].max()/1e6, 2), 'M')

In [ ]:
# ── KPI Chart 2: Category KPIs ─────────────────────────────
cat = df.groupby('category').agg(
    revenue=('revenue','sum'),
    profit=('gross_profit','sum'),
    units=('units_sold','sum'),
    margin=('profit_margin_pct','mean'),
    our_price=('current_price','mean'),
    comp_price=('comp_avg_price','mean'),
    rating=('product_rating','mean')
).reset_index().sort_values('revenue', ascending=False)

colors = ['#6366f1','#06b6d4','#10b981','#ec4899','#f59e0b']
cat_labels = cat['category'].str.split().str[0].tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('KPI 3 — Category Performance', fontsize=13, fontweight='bold')

axes[0,0].bar(cat_labels, cat['revenue']/1e6, color=colors)
axes[0,0].set_title('Revenue ($M)'); axes[0,0].grid(alpha=0.3, axis='y')

axes[0,1].bar(cat_labels, cat['margin'], color=colors)
axes[0,1].set_title('Avg Gross Margin (%)'); axes[0,1].grid(alpha=0.3, axis='y')

axes[1,0].pie(cat['profit'], labels=cat_labels, colors=colors,
              autopct='%1.1f%%', startangle=140)
axes[1,0].set_title('Profit Share by Category')

axes[1,1].bar(cat_labels, cat['rating'], color=colors)
axes[1,1].set_ylim(4.3, 4.9); axes[1,1].set_title('Avg Rating / 5.0')
axes[1,1].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('kpi_category.png', dpi=150, bbox_inches='tight'); plt.show()
print(cat[['category','revenue','margin','rating']].to_string(index=False))

In [ ]:
# ── KPI Chart 3: Competitor Price Gap (KPI 4) ──────────────
cat['price_advantage'] = cat['comp_price'] - cat['our_price']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('KPI 4 — Competitor Price Analysis', fontsize=13, fontweight='bold')

# Our price vs competitor
x = range(len(cat))
axes[0].bar([i-0.2 for i in x], cat['our_price'],      0.38, label='Our Price',      color='#6366f1aa')
axes[0].bar([i+0.2 for i in x], cat['comp_price'],     0.38, label='Comp Avg Price', color='#ef444466')
axes[0].set_xticks(x); axes[0].set_xticklabels(cat_labels, rotation=10)
axes[0].set_ylabel('Price ($)'); axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')
axes[0].set_title('Our Price vs Competitor Average')

# Price advantage
axes[1].bar(cat_labels, cat['price_advantage'], color='#10b981')
axes[1].set_title('Price Advantage — How Much Cheaper Are We? ($)')
axes[1].set_ylabel('$ Below Competitor'); axes[1].grid(alpha=0.3, axis='y')
for bar, v in zip(axes[1].patches, cat['price_advantage']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                 f'+${v:.2f}', ha='center', fontsize=10, fontweight='bold', color='#047857')

plt.tight_layout(); plt.savefig('kpi_competitor.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── KPI Chart 4: Promotion Impact (KPI 5) ─────────────────
promo = df.groupby('is_promotion').agg(
    revenue=('revenue','sum'),
    units=('units_sold','sum'),
    margin=('profit_margin_pct','mean')
).reset_index()
promo['label'] = promo['is_promotion'].map({0:'No Promo', 1:'With Promo'})

# KPI Chart 5: Day of Week Demand (KPI 6)
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('day_of_week')['units_sold'].mean().reindex(dow_order).reset_index()
dow.columns = ['day','avg_units']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('KPI 5 & 6 — Promotion Impact & Weekly Demand', fontsize=13, fontweight='bold')

pcolors = ['#33415588','#6366f1cc']
for ax, col, label, fmt in zip(
    axes[:2],
    ['revenue','units'],
    ['Revenue ($M)','Units Sold'],
    ['M','K']
):
    vals = promo[col]/(1e6 if fmt=='M' else 1e3)
    bars = ax.bar(promo['label'], vals, color=pcolors)
    ax.set_title(f'{label} — Promo vs No Promo'); ax.grid(alpha=0.3, axis='y')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+vals.max()*0.01,
                f'${v:.1f}{fmt}' if fmt=='M' else f'{v:.0f}K', ha='center', fontweight='bold')

dow_colors = ['#6366f1bb' if d not in ['Saturday','Sunday'] else '#ec4899bb' for d in dow['day']]
bars = axes[2].bar([d[:3] for d in dow['day']], dow['avg_units'], color=dow_colors)
axes[2].set_title('Avg Daily Units by Day of Week'); axes[2].grid(alpha=0.3, axis='y')
for b, v in zip(bars, dow['avg_units']):
    axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}',
                 ha='center', fontsize=9, fontweight='bold')

plt.tight_layout(); plt.savefig('kpi_promo_weekly.png', dpi=150, bbox_inches='tight'); plt.show()

# Save KPIs to JSON
os.makedirs('outputs', exist_ok=True)
kpis = {
    'total_revenue': round(float(total_rev), 2),
    'total_profit':  round(float(total_profit), 2),
    'avg_margin':    round(float(avg_margin), 2),
    'total_units':   int(total_units),
    'avg_price':     round(float(avg_price), 2),
    'avg_competitor_price': round(float(avg_comp), 2),
    'price_advantage': round(float(avg_comp - avg_price), 2),
    'avg_rating':    round(float(avg_rating), 2),
    'monthly': monthly[['month_name','revenue','profit','units']].to_dict('records'),
    'categories': cat[['category','revenue','margin','rating']].to_dict('records')
}
with open('outputs/kpis.json','w') as f: json.dump(kpis, f, indent=2, default=str)
print('✅ KPIs saved to outputs/kpis.json')
print(json.dumps({k:v for k,v in kpis.items() if k not in ['monthly','categories']}, indent=2))

---
# 🤖 TASK 1 — ML MODEL TRAINING
We train 5 models and compare them. Each model is in its own cell.

**Helper function** — run this first, then run each model cell.

In [ ]:
# Helper function to evaluate any model
def evaluate(name, model, X_tr, X_te, y_tr, y_te, features, color='#6366f1', save=True):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2   = r2_score(y_te, y_pred)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))

    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(f'  R² Score : {r2:.4f}  (1.0 = perfect)')
    print(f'  MAE      : ${mae:.2f}  (avg $ error per prediction)')
    print(f'  RMSE     : ${rmse:.2f}  (penalises large errors more)')

    # Actual vs predicted chart
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{name} — Evaluation', fontsize=12, fontweight='bold')

    axes[0].scatter(y_te, y_pred, alpha=0.3, color=color, s=12)
    axes[0].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=2)
    axes[0].set_title('Actual vs Predicted Price')
    axes[0].set_xlabel('Actual ($)'); axes[0].set_ylabel('Predicted ($)')

    if hasattr(model, 'feature_importances_'):
        fi = pd.Series(model.feature_importances_, index=features).nlargest(8)
        fi.sort_values().plot(kind='barh', ax=axes[1], color=color)
        axes[1].set_title('Top 8 Feature Importances')
    else:
        residuals = y_te - y_pred
        axes[1].hist(residuals, bins=40, color=color, alpha=0.7, edgecolor='white')
        axes[1].set_title('Residuals (Actual − Predicted)')
        axes[1].set_xlabel('Residual ($)')

    plt.tight_layout()
    if save: plt.savefig(f'outputs/{name.replace(" ","_").lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    return {'Model': name, 'R²': round(r2,4), 'MAE($)': round(mae,2), 'RMSE($)': round(rmse,2), 'estimator': model}

os.makedirs('outputs', exist_ok=True)
results = []
print('✅ Helper function ready! Now run each model cell below.')

## 📈 MODEL 1 — Linear Regression (Baseline)
**Simplest model.** Draws a straight line through data to make predictions. Used as baseline to measure how much better the other models are.

In [ ]:
m1 = evaluate('Model 1 — Linear Regression',
              LinearRegression(),
              Xp_tr, Xp_te, yp_tr, yp_te, price_features, '#94a3b8')
results.append({k:v for k,v in m1.items() if k != 'estimator'})

## 🌳 MODEL 2 — Decision Tree
**Asks yes/no questions** about the data to reach a prediction. Like a flowchart — easy to understand but can overfit.

In [ ]:
m2 = evaluate('Model 2 — Decision Tree',
              DecisionTreeRegressor(max_depth=8, random_state=42),
              Xp_tr, Xp_te, yp_tr, yp_te, price_features, '#f59e0b')
results.append({k:v for k,v in m2.items() if k != 'estimator'})

## 🌲 MODEL 3 — Random Forest
**Builds 200 Decision Trees** and averages their predictions. Much more accurate and stable than a single tree.

In [ ]:
m3 = evaluate('Model 3 — Random Forest',
              RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1),
              Xp_tr, Xp_te, yp_tr, yp_te, price_features, '#06b6d4')
results.append({k:v for k,v in m3.items() if k != 'estimator'})

## ⚡ MODEL 4 — XGBoost
**Builds trees one by one**, each fixing the mistakes of the previous. Very fast, very accurate. Industry standard.

In [ ]:
m4 = evaluate('Model 4 — XGBoost',
              XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                           subsample=0.8, random_state=42, verbosity=0),
              Xp_tr, Xp_te, yp_tr, yp_te, price_features, '#6366f1')
results.append({k:v for k,v in m4.items() if k != 'estimator'})

## 🏆 MODEL 5 — LightGBM (WINNER)
**Microsoft's fastest gradient booster.** Builds trees leaf-by-leaf (fixing the worst errors first). Achieves the best accuracy in the shortest time.

In [ ]:
lgbm_model = LGBMRegressor(n_estimators=400, learning_rate=0.1, num_leaves=31, random_state=42, verbose=-1)
m5 = evaluate('Model 5 — LightGBM (WINNER)',
              lgbm_model,
              Xp_tr, Xp_te, yp_tr, yp_te, price_features, '#10b981')
results.append({k:v for k,v in m5.items() if k != 'estimator'})

# Save the winner
joblib.dump({'model': lgbm_model, 'features': price_features,
             'le_cat': le_cat, 'le_chan': le_chan, 'le_dow': le_dow}, 'outputs/price_model.pkl')
print('\n✅ LightGBM saved as outputs/price_model.pkl')

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPARISON TABLE + CHART — All 5 Models
# ═══════════════════════════════════════════════════════

results_df = pd.DataFrame(results)
print('\n' + '='*55)
print('  PRICE PREDICTION — ALL 5 MODELS COMPARISON')
print('='*55)
print(results_df.to_string(index=False))
print('\n🏆 WINNER:', results_df.sort_values('RMSE($)').iloc[0]['Model'])

# Comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('All 5 Models — Price Prediction Comparison', fontsize=13, fontweight='bold')
mnames = [m.split('—')[1].strip().split(' (')[0] for m in results_df['Model']]
mcolors = ['#94a3b8','#f59e0b','#06b6d4','#6366f1','#10b981']

for ax, col, title in zip(axes,
    ['R²','MAE($)','RMSE($)'],
    ['R² Score (Higher=Better)','MAE $ (Lower=Better)','RMSE $ (Lower=Better)']):
    bars = ax.bar(mnames, results_df[col], color=mcolors, edgecolor='white')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.grid(alpha=0.3, axis='y')
    for b, v in zip(bars, results_df[col]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(results_df[col])*0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# DEMAND FORECASTING MODEL — XGBoost (Best for Demand)
# ═══════════════════════════════════════════════════════

demand_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                             subsample=0.8, random_state=42, verbosity=0)
demand_model.fit(Xd_tr, yd_tr)
yd_pred = demand_model.predict(Xd_te)

r2_d   = r2_score(yd_te, yd_pred)
mae_d  = mean_absolute_error(yd_te, yd_pred)
rmse_d = np.sqrt(mean_squared_error(yd_te, yd_pred))

print('DEMAND FORECASTING — XGBOOST (WINNER)')
print(f'  R² Score : {r2_d:.4f}')
print(f'  MAE      : {mae_d:.2f} units/day')
print(f'  RMSE     : {rmse_d:.2f} units/day')
print(f'  Meaning  : Predictions are within ~{mae_d:.0f} units of actual daily demand')

# Actual vs predicted demand
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(yd_te, yd_pred, alpha=0.3, color='#6366f1', s=15)
ax.plot([yd_te.min(), yd_te.max()], [yd_te.min(), yd_te.max()], 'r--', lw=2)
ax.set_title(f'XGBoost Demand Model — Actual vs Predicted\nR²={r2_d:.4f} | MAE={mae_d:.1f} units')
ax.set_xlabel('Actual Units Sold'); ax.set_ylabel('Predicted Units Sold')
plt.tight_layout(); plt.savefig('outputs/demand_model.png', dpi=150, bbox_inches='tight'); plt.show()

joblib.dump({'model': demand_model, 'features': demand_features}, 'outputs/demand_model.pkl')
print('✅ Demand model saved as outputs/demand_model.pkl')

---
# 🔌 TASK 3 — Connect to Groq LLM via API Key

**Get your FREE API key:** Go to https://console.groq.com → Sign Up → API Keys → Create New Key

**Model used:** `llama-3.3-70b-versatile` — Meta's 70B parameter model, ultra-fast (~0.5 sec)

In [ ]:
from groq import Groq

# ⬇️ Paste your Groq API key here (free from console.groq.com)
GROQ_API_KEY = 'gsk_your_key_here'

# Create connection
client = Groq(api_key=GROQ_API_KEY)

# Test connection
test = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'You are a helpful AI pricing analyst.'},
        {'role': 'user',   'content': 'Confirm connection for PricePilot AI Milestone 2.'}
    ],
    temperature=0.3, max_tokens=60
)

print('✅ Groq LLM Connected Successfully!')
print(f'   Response  : {test.choices[0].message.content}')
print(f'   Model     : {test.model}')
print(f'   Tokens used: {test.usage.total_tokens}')
print(f'   Free tier : ~14,400 requests/day')

---
# 💡 TASK 4 — LLM Integration: Send KPI Data → Get AI Insights

We send the KPI numbers extracted in Task 2 to the Groq LLM and get AI-generated pricing recommendations.

In [ ]:
with open('outputs/kpis.json') as f:
    kpis = json.load(f)

# ═══ INSIGHT 1: Demand Analysis ════════════════════════════
def demand_insight(product, category, price, demand, comp_price, margin):
    prompt = f"""
    You are a senior retail pricing analyst. Give clear, simple advice.

    Product: {product}
    Category: {category}
    Our Price: ${price} | Daily Demand: {demand} units | Comp Price: ${comp_price} | Margin: {margin}%

    Business context: Total revenue ${kpis['total_revenue']:,.0f} | Avg margin {kpis['avg_margin']}%
    We are ${kpis['price_advantage']:.2f} cheaper than competitors overall.

    Answer in 3 short bullet points: (1) Demand assessment (2) Pricing signal (3) Recommended action.
    """
    r = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':prompt}],
        temperature=0.4, max_tokens=220
    )
    return r.choices[0].message.content

# ═══ INSIGHT 2: Price Optimization ════════════════════════
def price_optimization(product, our_price, comp1, comp2, comp3, elasticity):
    prompt = f"""
    As a pricing strategist, recommend an optimal price.
    Product: {product} | Our Price: ${our_price} | Competitors: ${comp1}, ${comp2}, ${comp3}
    Price elasticity: {elasticity} (more negative = more price-sensitive customers)
    Give: recommended price in $, expected revenue change %, and risk level (Low/Medium/High). Under 100 words.
    """
    r = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':prompt}],
        temperature=0.3, max_tokens=180
    )
    return r.choices[0].message.content

# ═══ INSIGHT 3: Competitor Report ═════════════════════════
def competitor_report():
    cats_text = '\n'.join([f"  {c['category']}: Margin {c['margin']:.1f}%" for c in kpis['categories']])
    prompt = f"""
    Competitive pricing report for PricePilot AI:
    We are ${kpis['price_advantage']:.2f} cheaper than competitors on average across all categories.
    Category margins:\n{cats_text}
    Total revenue: ${kpis['total_revenue']:,.0f} | Average margin: {kpis['avg_margin']}%
    Give 3 bullet points: where to raise prices, where to maintain, and the overall risk. Under 120 words.
    """
    r = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':prompt}],
        temperature=0.4, max_tokens=200
    )
    return r.choices[0].message.content

# ═══ INSIGHT 4: Seasonal Strategy ═════════════════════════
def seasonal_strategy():
    months_text = '\n'.join([f"  {m['month_name']}: ${m['revenue']:,.0f}" for m in kpis['monthly']])
    prompt = f"""
    Based on 2025 monthly revenue data:
{months_text}
    Facts: Holiday demand +275%, Promotions drop margin 7 pts, Mon-Tue peak, Fri lowest.
    Give 3 bullet points: best promotion months, best price-raise months, holiday prep. Under 120 words.
    """
    r = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':prompt}],
        temperature=0.5, max_tokens=200
    )
    return r.choices[0].message.content

# ════ RUN ALL 4 INSIGHTS ════════════════════════════════════
print('INSIGHT 1 — DEMAND ANALYSIS')
print('-'*50)
print(demand_insight('Aura Pro Headphones','Electronics',187.50,40.7,196.43,41.09))

print('\nINSIGHT 2 — PRICE OPTIMIZATION')
print('-'*50)
print(price_optimization('Aura Pro Headphones',187.50,199.99,195.00,194.31,-1.85))

print('\nINSIGHT 3 — COMPETITOR REPORT')
print('-'*50)
print(competitor_report())

print('\nINSIGHT 4 — SEASONAL STRATEGY')
print('-'*50)
print(seasonal_strategy())

print('\n✅ All 4 AI insights generated!')

---
# 📥 Download All Output Files
Run this cell to download all generated files to your computer.

In [ ]:
from google.colab import files
import zipfile

# Create zip of all outputs
with zipfile.ZipFile('PricePilot_Milestone2_Outputs.zip', 'w') as zf:
    for f in os.listdir('outputs'):
        zf.write(f'outputs/{f}', f)

files.download('PricePilot_Milestone2_Outputs.zip')
print('✅ Download started!')
print('Files included:')
for f in sorted(os.listdir('outputs')):
    size = os.path.getsize(f'outputs/{f}')
    print(f'  {f:45s} ({size/1024:.1f} KB)')